# EXP-02: Oversampling of images with small symbols

**Hypothesis.** In the EXP-00 error analysis (full validation set, 240,867 tokens, 1,604 errors) the largest error source is omission of small symbols that sit next to notes: 700 deletions, 336 of them `accidental.natural` (recall 0.768), plus `accidental.flat` (110), slurs (77) and `gracenote.sixteenth` (recall 0.747; shape-only recall 0.967, so mostly position errors). EXP-01 showed that a finer horizontal resolution alone does not fix this (CER 0.0073 vs 0.0073, 6.33 h vs 3.94 h). If the problem is under-exposure, drawing the training images that contain these symbols more often should improve their recall without changing the architecture. The mechanism is a hypothesis: the error analysis only shows where the errors are, not why.

**Single change vs EXP-00.** Training images are drawn **with replacement** and with unequal weights. Each epoch draws `round(1.10 x 70,138)` images with probability proportional to a weight: `K = 3` for images that contain at least one token of a target family, `1` for the rest. Only the first 1,830 batches of that stream are used (the extra 10 % guarantees that the bucketed pipeline always yields enough full batches). Steps per epoch (1,830), total steps (14,640), LR schedule, batch policy, model (1,896,407 parameters), seed, initial weights, optimizer, 8 epochs, fixed 2,000-sample validation subset and best-model criterion are identical to EXP-00. Validation and test are untouched.

**Target families** (`TARGET_FAMILIES`): `accidental.natural`, `slur.start`, `slur.end`, `gracenote.sixteenth`, `gracenote.thirty_second`. Left out on purpose: `accidental.flat` (very frequent: including it would enlarge the target set and dilute the oversampling) and `barline` (its errors come from the final-barline label inconsistency, not from visual difficulty).

**Hyperparameter.** `OVERSAMPLE_K = 3`. Only this value is run; it is not tuned.

**Success criteria** (full validation set, measured in `04_evaluation.ipynb`, section 11, against EXP-00):
- `accidental.natural` recall >= 0.85 (EXP-00: 0.768).
- Net errors (substitutions + deletions + insertions) in the oversampled families at least 25 % lower than EXP-00. Insertions are included so that a gain in recall bought with spurious symbols does not count.
- General performance maintained: full-validation CER <= 0.0074 (at most 10 % above EXP-00's 0.0067; with one run per experiment, smaller gaps cannot be told apart from noise).
- Secondary / diagnostic: CER ignoring a trailing `barline-L1`, and CER on validation images with and without target tokens.

**Reference (EXP-00).** Subset CER 0.0073 (best epoch 8), full-validation CER 0.0067, seq_acc 0.8805, cer_shape 0.0054, 3.94 h of training.

**Known costs and limitations.**
- Images without target tokens are drawn less often (the expected exposure is printed in section 6).
- No augmentation: a repeated image is identical every time, so only dropout limits memorization of the oversampled images. Augmentation is the natural complement and is left as a conditional future experiment (EXP-03).
- One run per experiment; run-to-run variation is not measured.

**After a disconnect:** `Runtime -> Run all`. Section 8 resumes automatically from the last checkpoint in `artifacts/checkpoints/exp02/`. The composition of each epoch is a pure function of (seed, epoch), so a resumed epoch uses the same sample of images (in a new order, as in EXP-00/01).

## 1. Setup

### 1.1. Imports

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
from itertools import islice
import json, math, os, time
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from PIL import Image

### 1.2. Drive mount, paths and extraction

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = Path('/content/drive/MyDrive/NoteVision')
RAW_DIR = PROJECT_DIR / 'data_raw'
ARTIFACTS_DIR = PROJECT_DIR / 'artifacts'
LOCAL_DIR = Path('/content/primus')
ARCHIVE = RAW_DIR / 'primus.tgz'
EXPECTED_SAMPLES = 87_678
SEED = 42

def count_samples(root):
    n = 0
    for pkg in ('package_aa', 'package_ab'):
        p = root / pkg
        if p.exists():
            n += sum(1 for d in p.iterdir() if d.is_dir() and not d.name.startswith('._'))
    return n

if count_samples(LOCAL_DIR) != EXPECTED_SAMPLES:
    LOCAL_DIR.mkdir(parents=True, exist_ok=True)
    !tar -xzf $ARCHIVE -C $LOCAL_DIR

n = count_samples(LOCAL_DIR)
print('Samples on disk:', n)
assert n == EXPECTED_SAMPLES, 'Incomplete extraction'

Mounted at /content/drive
Samples on disk: 87678


### 1.3. Preprocessing functions and metrics

In [ ]:
IMG_HEIGHT = 128

def load_vocab(vocab_csv):
    """Return (tokens, token_to_id, n_tokens)"""
    df = pd.read_csv(vocab_csv, dtype=str, keep_default_na=False)
    tokens = df['token'].tolist()
    return tokens, {t: i for i, t in enumerate(tokens)}, len(tokens)

def encode_labels(df, token_to_id, max_len):
    """Dense int32 array [N, max_len] (zero padded) and true lengths [N]."""
    labels = np.zeros((len(df), max_len), dtype=np.int32)
    lengths = np.zeros(len(df), dtype=np.int32)
    for i, seq in enumerate(df['label']):
        ids = [token_to_id[t] for t in seq.split()]
        labels[i, :len(ids)] = ids
        lengths[i] = len(ids)
    return labels, lengths

def decode_ids(ids, tokens):
    return ' '.join(tokens[i] for i in ids)

def image_path(root, package, sid):
    return Path(root) / package / sid / f'{sid}.png'

def load_image(path):
    raw = tf.io.read_file(path)
    img = tf.io.decode_png(raw, channels=3) # palette (mode P) -> RGB
    gray = tf.image.rgb_to_grayscale(img) # uint8 [H, W, 1]
    h = tf.cast(tf.shape(gray)[0], tf.float32)
    w = tf.cast(tf.shape(gray)[1], tf.float32)
    new_w = tf.maximum(1, tf.cast(tf.round(w * IMG_HEIGHT / h), tf.int32))
    gray = tf.image.resize(tf.cast(gray, tf.float32), (IMG_HEIGHT, new_w),
                           method='bilinear', antialias=True)
    gray = tf.clip_by_value(gray, 0.0, 255.0)
    return 1.0 - gray / 255.0, new_w # ink = 1, background/padding = 0

def align(ref, hyp):
    n, m = len(ref), len(hyp)
    d = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1):
        d[i][0] = i
    for j in range(m + 1):
        d[0][j] = j
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            d[i][j] = min(d[i-1][j] + 1, d[i][j-1] + 1,
                          d[i-1][j-1] + (ref[i-1] != hyp[j-1]))
    ops, i, j = [], n, m
    while i > 0 or j > 0:
        if i > 0 and j > 0 and d[i][j] == d[i-1][j-1] + (ref[i-1] != hyp[j-1]):
            ops.append(('hit' if ref[i-1] == hyp[j-1] else 'sub', ref[i-1]))
            i, j = i - 1, j - 1
        elif i > 0 and d[i][j] == d[i-1][j] + 1:
            ops.append(('del', ref[i-1]))
            i -= 1
        else:
            ops.append(('ins', None))
            j -= 1
    return d[n][m], ops

def evaluate_sequences(refs, hyps, families):
    tot_ref = tot_err = tot_err_shape = exact = 0
    per_fam = defaultdict(Counter)
    for r, h in zip(refs, hyps):
        dist, ops = align(r, h)
        tot_err += dist
        tot_ref += len(r)
        exact += int(r == h)
        tot_err_shape += align([families[i] for i in r], [families[i] for i in h])[0]
        for op, tok in ops:
            if tok is not None:
                per_fam[families[tok]][op] += 1
    metrics = {'cer': tot_err / tot_ref, 'seq_acc': exact / len(refs),
               'cer_shape': tot_err_shape / tot_ref,
               'n_sequences': len(refs), 'n_tokens': tot_ref}
    rows = []
    for fam, c in per_fam.items():
        support = c['hit'] + c['sub'] + c['del']
        rows.append({'family': fam, 'support': support, 'recall': c['hit'] / support})
    fam_df = pd.DataFrame(rows).sort_values('support', ascending=False).reset_index(drop=True)
    return metrics, fam_df

# Sanity checks
_fam = ['a', 'a', 'b']
assert align([0, 1, 2], [0, 1, 2])[0] == 0
assert align([0, 1, 2], [0, 2])[0] == 1
assert align([0, 1, 2], [0, 1, 1])[0] == 1
_m, _ = evaluate_sequences([[0, 1, 2]], [[0, 0, 2]], _fam)
assert abs(_m['cer'] - 1/3) < 1e-9 and _m['cer_shape'] == 0.0 and _m['seq_acc'] == 0.0
print('Metric sanity checks passed.')

Metric sanity checks passed.


### 1.4. Bootstrap

In [ ]:
parts = {s: pd.read_csv(ARTIFACTS_DIR / f'split_{s}.csv', dtype=str)
         for s in ('train', 'val', 'test')}
for d in parts.values():
    d['seq_len'] = d['seq_len'].astype(int)

tokens, token_to_id, _ = load_vocab(ARTIFACTS_DIR / 'vocab_agnostic.csv')
cfg = json.loads((ARTIFACTS_DIR / 'preprocessing_config.json').read_text())
summ = json.loads((ARTIFACTS_DIR / 'split_summary.json').read_text())

# Guard against drift between notebooks
assert IMG_HEIGHT == cfg['img_height'] and len(tokens) == cfg['vocab_size']
assert {s: len(d) for s, d in parts.items()} == {s: summ[f'n_{s}'] for s in parts}
MAX_LABEL_LEN = cfg['max_label_len']
print('Bootstrap OK:', {s: len(d) for s, d in parts.items()})

Bootstrap OK: {'train': 70138, 'val': 8772, 'test': 8768}


## 2. GPU and Seed

In [ ]:
gpus = tf.config.list_physical_devices('GPU')
print('GPUs:', gpus)
assert gpus
for g in gpus:
    tf.config.experimental.set_memory_growth(g, True)
tf.keras.utils.set_random_seed(SEED)

GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 3. Width-Bucketed Pipeline

In [ ]:
# Boundaries every 100 px up to 2200, then coarser. Each batch is padded to (boundary - 1),
# so the whole training run sees only ~25 distinct input shapes.
BOUNDS = list(range(600, 2201, 100)) + [2600, 3000, 3600]
MAX_PIXELS = 48000 # batch_size * padded_width budget (~36 images at 1266 px, ~5 GB peak)
MAX_BATCH = 64
BATCH_SIZES = [max(4, min(MAX_BATCH, MAX_PIXELS // b)) for b in BOUNDS] + [4] # last bucket (>= 3600) stays empty
print('Buckets:', len(BOUNDS) + 1, '| batch size per bucket:', BATCH_SIZES)

# Safety: the widest resized image must fit under the top boundary
sz = pd.read_csv(ARTIFACTS_DIR / 'image_sizes.csv')
max_w = int(np.round(sz.width * IMG_HEIGHT / sz.height).max())
assert max_w < BOUNDS[-1], f'widest image ({max_w}) does not fit in the buckets'


def build_dataset_bucketed(df, root, token_to_id, max_label_len, training=False, seed=42):
    labels, lengths = encode_labels(df, token_to_id, max_label_len)
    paths = [str(image_path(root, p, i)) for p, i in zip(df['package'], df['id'])]
    ds = tf.data.Dataset.from_tensor_slices((paths, labels, lengths))
    if training:
        ds = ds.shuffle(len(paths), seed=seed, reshuffle_each_iteration=True)

    def _map(path, label, length):
        image, width = load_image(path)
        return {'image': image, 'width': width, 'label': label, 'label_length': length}

    ds = ds.map(_map, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.bucket_by_sequence_length(
        element_length_func=lambda e: e['width'],
        bucket_boundaries=BOUNDS, bucket_batch_sizes=BATCH_SIZES,
        padded_shapes={'image': [IMG_HEIGHT, None, 1], 'width': [],
                       'label': [max_label_len], 'label_length': []},
        padding_values={'image': tf.constant(0.0), 'width': tf.constant(0, tf.int32),
                        'label': tf.constant(0, tf.int32),
                        'label_length': tf.constant(0, tf.int32)},
        pad_to_bucket_boundary=True,
        drop_remainder=training) # fixed batch shape in training; keep every sample in val/test
    return ds.prefetch(tf.data.AUTOTUNE)

Buckets: 21 | batch size per bucket: [64, 64, 60, 53, 48, 43, 40, 36, 34, 32, 30, 28, 26, 25, 24, 22, 21, 18, 16, 13, 4]


## 4. CRNN Model (x8)

Same architecture as EXP-00: four convolutional blocks, horizontal reduction x8, two BiLSTM layers of 128 units per direction, 1,896,407 trainable parameters. The smoke test below asserts the parameter count and the shapes.

In [ ]:
REDUCTION = 8 # horizontal downsampling (identical to EXP-00)
CTC_BLANK = 0 # class 0 = blank; token id i -> class i + 1
NUM_CLASSES = len(tokens) + 1 # 758 tokens + CTC blank
families = [t.split('-')[0] for t in tokens]

MODEL_CFG = {'lstm_units': 128, 'dropout': 0.25}

class CRNN(keras.Model):
    def __init__(self, num_classes, lstm_units=128, dropout=0.25, **kw):
        super().__init__(**kw)
        filters = (32, 64, 128, 256)
        pools = ((2, 2), (2, 2), (2, 2), (2, 1)) # width /2 three times -> x8; last pool only in height
        self.convs = [layers.Conv2D(f, 3, padding='same', use_bias=False) for f in filters]
        self.bns = [layers.BatchNormalization() for _ in filters]
        self.pools = [layers.MaxPool2D(p) for p in pools]
        self.proj = layers.Dense(256, activation='relu')
        self.drop = layers.Dropout(dropout)
        self.rnn1 = layers.Bidirectional(layers.LSTM(lstm_units, return_sequences=True))
        self.rnn2 = layers.Bidirectional(layers.LSTM(lstm_units, return_sequences=True))
        self.classifier = layers.Dense(num_classes)

    def call(self, inputs, training=False):
        x, widths = inputs['image'], inputs['width']
        for conv, bn, pool in zip(self.convs, self.bns, self.pools):
            x = pool(tf.nn.relu(bn(conv(x), training=training)))
        x = tf.transpose(x, [0, 2, 1, 3])
        s = tf.shape(x)
        x = tf.reshape(x, [s[0], s[1], x.shape[2] * x.shape[3]])
        x = self.drop(self.proj(x), training=training)
        logit_len = widths // REDUCTION
        mask = tf.sequence_mask(logit_len, maxlen=s[1]) # ignore padding in the BiLSTM
        x = self.rnn1(x, mask=mask, training=training)
        x = self.rnn2(self.drop(x, training=training), mask=mask, training=training)
        return self.classifier(self.drop(x, training=training)), logit_len

def build_model(cfg):
    model = CRNN(NUM_CLASSES, lstm_units=cfg['lstm_units'], dropout=cfg['dropout'])
    model({'image': tf.zeros([1, IMG_HEIGHT, 256, 1]), 'width': tf.constant([256])})  # build weights
    return model

### 4.1. Shape smoke test

In [ ]:
tf.keras.utils.set_random_seed(SEED)
smoke = build_model(MODEL_CFG)

# 1) Shape logic on a synthetic input (padded width 600; real widths 599 and 300)
out, ln = smoke({'image': tf.zeros([2, IMG_HEIGHT, 600, 1]), 'width': tf.constant([599, 300])},
                training=False)
assert tuple(out.shape) == (2, 600 // REDUCTION, NUM_CLASSES)
assert ln.numpy().tolist() == [599 // REDUCTION, 300 // REDUCTION]

# 2) One real bucketed training batch
b = next(iter(build_dataset_bucketed(parts['train'], LOCAL_DIR, token_to_id, MAX_LABEL_LEN,
                                     training=True, seed=SEED)))
logits, logit_len = smoke({'image': b['image'], 'width': b['width']}, training=False)
print('image:', tuple(b['image'].shape), '| logits:', tuple(logits.shape))
assert logits.shape[1] == b['image'].shape[2] // REDUCTION
assert logits.shape[-1] == NUM_CLASSES
assert bool(tf.reduce_all(logit_len <= logits.shape[1]))
assert bool(tf.reduce_all(logit_len >= b['label_length'])) # CTC needs T >= label length

n_params = sum(int(np.prod(v.shape)) for v in smoke.trainable_variables)
print('Trainable params:', f'{n_params:,}')
assert n_params == 1_896_407, 'parameter count differs from EXP-00'
del smoke

image: (40, 128, 1199, 1) | logits: (40, 149, 759)
Trainable params: 1,896,407


## 5. CTC Loss, Training Step and Greedy Decoding

In [ ]:
def ctc_loss_fn(labels, label_len, logits, logit_len):
    loss = tf.nn.ctc_loss(labels=labels + 1, # token id i -> class i + 1
                          logits=tf.cast(logits, tf.float32),
                          label_length=label_len, logit_length=logit_len,
                          logits_time_major=False, blank_index=CTC_BLANK)
    return tf.reduce_mean(loss)

def make_steps(model, optimizer):
    """Build tf.functions bound to THIS model and optimizer (call again for each new model)."""
    @tf.function(input_signature=[
        tf.TensorSpec([None, IMG_HEIGHT, None, 1], tf.float32),
        tf.TensorSpec([None], tf.int32),
        tf.TensorSpec([None, MAX_LABEL_LEN], tf.int32),
        tf.TensorSpec([None], tf.int32)])
    def train_step(image, width, label, label_len):
        with tf.GradientTape() as tape:
            logits, logit_len = model({'image': image, 'width': width}, training=True)
            loss = ctc_loss_fn(label, label_len, logits, logit_len)
        grads = tape.gradient(loss, model.trainable_variables)
        optimizer.apply_gradients(zip(grads, model.trainable_variables))
        return loss

    @tf.function(input_signature=[
        tf.TensorSpec([None, IMG_HEIGHT, None, 1], tf.float32),
        tf.TensorSpec([None], tf.int32)])
    def infer(image, width):
        return model({'image': image, 'width': width}, training=False)

    return train_step, infer

def greedy_decode(logits, lengths, blank=CTC_BLANK):
    """argmax per step -> collapse repeats -> drop blanks -> back to token ids 0..757."""
    ids = np.argmax(logits, axis=-1)
    out = []
    for seq, n in zip(ids, lengths):
        seq = seq[:n]
        changed = np.concatenate([[True], seq[1:] != seq[:-1]])
        seq = seq[changed]
        out.append((seq[seq != blank] - 1).tolist())
    return out

def predict(batches):
    refs, hyps = [], []
    for b in batches:
        logits, logit_len = infer(b['image'], b['width'])
        hyps += greedy_decode(logits.numpy(), logit_len.numpy())
        lab, ln = b['label'].numpy(), b['label_length'].numpy()
        refs += [lab[i, :ln[i]].tolist() for i in range(len(ln))]
    return refs, hyps

# Decode sanity check (no model): blank=0, token id i = class i + 1
def _fake(classes):
    z = np.zeros((1, len(classes), NUM_CLASSES), dtype=np.float32)
    for t, c in enumerate(classes):
        z[0, t, c] = 1.0
    return z

assert greedy_decode(_fake([0, 3, 3, 0, 5, 5, 0]), [7]) == [[2, 4]]
assert greedy_decode(_fake([3, 0, 3]), [3]) == [[2, 2]] # repeat separated by blank survives
print('Decode sanity checks passed.')

Decode sanity checks passed.


## 6. Oversampling

### 6.1. Target families, sampling weights and expected exposure

The sampler works on the training dataframe, before the width-bucketed pipeline of section 3, which is unchanged. The cell prints how much the draw distribution moves: the share of draws on images with target tokens, the expected number of times each image is drawn per epoch (EXP-00: exactly once), and the same figure per target family. If the share of target images is already high, K = 3 barely changes the distribution; if the exposure of the other images drops a lot, general performance is at risk.

In [ ]:
# Oversampling configuration (hyperparameters of EXP-02)
OVERSAMPLE_K = 3        # weight of images that contain a target-family token (1 = uniform sampling)
TARGET_FAMILIES = ['accidental.natural', 'slur.start', 'slur.end',
                   'gracenote.sixteenth', 'gracenote.thirty_second']
DRAW_FACTOR = 1.10      # draws per epoch = DRAW_FACTOR * len(train); only the first STEPS_PER_EPOCH batches are used

assert set(TARGET_FAMILIES) <= set(families), 'unknown target family'

def sampling_probs(train_df, cfg):
    """Draw probability per training image: weight K if it has a target-family token, 1 otherwise."""
    target_fams = set(cfg['target_families'])
    target_tokens = {t for t, f in zip(tokens, families) if f in target_fams}
    has_target = np.array([any(t in target_tokens for t in s.split()) for s in train_df['label']])
    w = np.where(has_target, float(cfg['oversample_k']), 1.0)
    return w / w.sum(), has_target

def sample_epoch_idx(n_rows, probs, cfg, epoch):
    """Row indices drawn with replacement for one epoch; a pure function of (seed, epoch)."""
    rng = np.random.default_rng(cfg['seed'] + 1000 * epoch)
    return rng.choice(n_rows, size=int(round(cfg['draw_factor'] * n_rows)), replace=True, p=probs)

def sample_epoch(train_df, probs, cfg, epoch):
    return train_df.iloc[sample_epoch_idx(len(train_df), probs, cfg, epoch)].reset_index(drop=True)

samp_cfg = {'seed': SEED, 'oversample_k': OVERSAMPLE_K, 'target_families': TARGET_FAMILIES,
            'draw_factor': DRAW_FACTOR}
train_df = parts['train']
n_train = len(train_df)
probs, has_target = sampling_probs(train_df, samp_cfg)
share_before, share_after = float(has_target.mean()), float(probs[has_target].sum())
exp_target = n_train * float(probs[has_target][0])    # expected draws per target image per epoch (~n_train draws are used)
exp_other = n_train * float(probs[~has_target][0])    # same for the other images

print(f'Train images with at least one target token: {int(has_target.sum()):,} of {n_train:,} ({share_before:.1%})')
print(f'Share of draws on those images: {share_before:.1%} -> {share_after:.1%} (K={OVERSAMPLE_K})')
print(f'Expected draws per image per epoch: target {exp_target:.2f}x | other {exp_other:.2f}x '
      f'(EXP-00: 1.00x for all) | other images not drawn at all in an epoch: ~{np.exp(-exp_other):.0%}')

rows = []
for fam in TARGET_FAMILIES:
    toks = {t for t, f in zip(tokens, families) if f == fam}
    cnt = np.array([sum(t in toks for t in s.split()) for s in train_df['label']])
    rows.append({'family': fam, 'train_tokens': int(cnt.sum()), 'images_with_family': int((cnt > 0).sum()),
                 'expected_exposure_vs_exp00': n_train * float((probs * cnt).sum() / cnt.sum())})
display(pd.DataFrame(rows).round(2))

Train images with at least one target token: 17,223 of 70,138 (24.6%)
Share of draws on those images: 24.6% -> 49.4% (K=3)
Expected draws per image per epoch: target 2.01x | other 0.67x (EXP-00: 1.00x for all) | other images not drawn at all in an epoch: ~51%


,family,train_tokens,images_with_family,expected_exposure_vs_exp00
0,accidental.natural,11409,8820,2.01
1,slur.start,10086,6975,2.01
2,slur.end,10086,6975,2.01
3,gracenote.sixteenth,5623,3074,2.01
4,gracenote.thirty_second,801,218,2.01


### 6.2. Sampler checks (no GPU needed)

In [ ]:
a = sample_epoch_idx(n_train, probs, samp_cfg, 1)
b = sample_epoch_idx(n_train, probs, samp_cfg, 1)
c = sample_epoch_idx(n_train, probs, samp_cfg, 2)
assert (a == b).all() and not (a == c).all()          # deterministic per epoch, a new draw every epoch
assert len(a) == round(DRAW_FACTOR * n_train)
assert len(np.unique(a)) < len(a)                     # sampling is with replacement
emp = float(has_target[a].mean())
assert abs(emp - share_after) < 0.01, (emp, share_after)   # empirical share matches the expected one
p_uniform, _ = sampling_probs(train_df, {**samp_cfg, 'oversample_k': 1})
assert np.allclose(p_uniform, 1.0 / n_train)          # K = 1 reduces to uniform sampling
epoch_df = sample_epoch(train_df, probs, samp_cfg, 1)
assert list(epoch_df.columns) == list(train_df.columns) and len(epoch_df) == len(a)
print(f'Sampler checks passed | empirical target share {emp:.1%} (expected {share_after:.1%}) | '
      f'unique images in one draw: {len(np.unique(a)):,} of {n_train:,}')

Sampler checks passed | empirical target share 49.5% (expected 49.4%) | unique images in one draw: 42,914 of 70,138


## 7. Training Setup

### 7.1. Config, schedule and steps per epoch

In [ ]:
CKPT_DIR = ARTIFACTS_DIR / 'checkpoints'
CKPT_DIR.mkdir(exist_ok=True)
RUN_DIR = CKPT_DIR / 'exp02'

class WarmupCosine(keras.optimizers.schedules.LearningRateSchedule):
    """Linear warmup, then cosine decay to lr_final; a pure function of the step."""
    def __init__(self, peak, final, warmup, total):
        super().__init__()
        self.peak, self.final = float(peak), float(final)
        self.warmup, self.total = int(warmup), int(total)

    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        warm = self.peak * (step + 1.0) / self.warmup
        prog = tf.clip_by_value((step - self.warmup) / float(max(1, self.total - self.warmup)), 0.0, 1.0)
        cos = self.final + 0.5 * (self.peak - self.final) * (1.0 + tf.cos(math.pi * prog))
        return tf.where(step < self.warmup, warm, cos)

    def get_config(self):
        return {'peak': self.peak, 'final': self.final, 'warmup': self.warmup, 'total': self.total}

def compute_steps_per_epoch(df):
    """Exact steps per epoch of the bucketed train pipeline (drop_remainder), from image headers only."""
    cache = ARTIFACTS_DIR / 'train_steps_per_epoch.json'
    sig = {'bounds': BOUNDS, 'batch_sizes': BATCH_SIZES, 'n_train': len(df)}
    if cache.exists():
        c = json.loads(cache.read_text())
        if c['sig'] == sig:
            return c['steps_per_epoch']
    widths = []
    for p, i in zip(df['package'], df['id']):
        with Image.open(image_path(LOCAL_DIR, p, i)) as im:
            w, h = im.size
        widths.append(max(1, int(np.round(np.float32(w) * IMG_HEIGHT / np.float32(h)))))
    bucket = np.searchsorted(BOUNDS, np.array(widths), side='right')
    counts = np.bincount(bucket, minlength=len(BATCH_SIZES))
    steps = int(sum(int(n) // bs for n, bs in zip(counts, BATCH_SIZES)))
    cache.write_text(json.dumps({'sig': sig, 'steps_per_epoch': steps, 'bucket_counts': counts.tolist()}))
    return steps

STEPS_PER_EPOCH = compute_steps_per_epoch(parts['train'])
assert STEPS_PER_EPOCH == 1830, 'steps per epoch differ from the EXP-00 budget'

CFG = {
    'exp_id': 'EXP-02', 'seed': SEED, 'epochs': 8, 'steps_per_epoch': STEPS_PER_EPOCH,
    'lr_peak': 1e-3, 'lr_final': 1e-5, 'warmup_steps': 200, 'clipnorm': 5.0,
    **MODEL_CFG, 'max_pixels': MAX_PIXELS, 'max_batch': MAX_BATCH,
    'val_subset_n': 2000, 'early_stop_from_epoch': 4, 'patience': 3,
    'oversample_k': OVERSAMPLE_K, 'target_families': list(TARGET_FAMILIES), 'draw_factor': DRAW_FACTOR,
}

# Fixed validation subset, shared by all experiments
val_sub = parts['val'].sample(CFG['val_subset_n'], random_state=SEED)
ids_path = ARTIFACTS_DIR / 'val_subset_ids.csv'
if ids_path.exists():
    saved = pd.read_csv(ids_path, dtype=str)
    assert saved['id'].tolist() == val_sub['id'].tolist(), 'val subset changed'
else:
    val_sub[['package', 'id']].to_csv(ids_path, index=False)

total = CFG['epochs'] * STEPS_PER_EPOCH
print(f'steps/epoch: {STEPS_PER_EPOCH} | total steps: {total} | '
      f'est. {total * 1.0 / 3600:.1f}-{total * 1.4 / 3600:.1f} h at 1.0-1.4 s/step '
      f'(EXP-00 took 3.94 h)')
print('Config:', CFG)

steps/epoch: 1830 | total steps: 14640 | est. 4.1-5.7 h at 1.0-1.4 s/step (EXP-00 took 3.94 h)
Config: {'exp_id': 'EXP-02', 'seed': 42, 'epochs': 8, 'steps_per_epoch': 1830, 'lr_peak': 0.001, 'lr_final': 1e-05, 'warmup_steps': 200, 'clipnorm': 5.0, 'lstm_units': 128, 'dropout': 0.25, 'max_pixels': 48000, 'max_batch': 64, 'val_subset_n': 2000, 'early_stop_from_epoch': 4, 'patience': 3, 'oversample_k': 3, 'target_families': ['accidental.natural', 'slur.start', 'slur.end', 'gracenote.sixteenth', 'gracenote.thirty_second'], 'draw_factor': 1.1}


### 7.2. Checkpoint helpers

In [ ]:
def write_json_atomic(path, obj):
    tmp = path.with_name(path.name + '.tmp')
    tmp.write_text(json.dumps(obj, indent=1))
    os.replace(tmp, path)

def save_ckpt(run_dir, model, optimizer, state):
    """Write to the alternate slot; the pointer (latest.json) is updated last."""
    run_dir.mkdir(parents=True, exist_ok=True)
    state['n_ckpts'] += 1
    slot = run_dir / f"slot{state['n_ckpts'] % 2}"
    slot.mkdir(exist_ok=True)
    tmp_w = slot / 'tmp.weights.h5'
    model.save_weights(tmp_w)
    os.replace(tmp_w, slot / 'model.weights.h5')
    tmp_o = slot / 'tmp_opt.npz'
    with open(tmp_o, 'wb') as f:
        np.savez(f, **{f'v{i}': v.numpy() for i, v in enumerate(optimizer.variables)})
    os.replace(tmp_o, slot / 'optimizer.npz')
    write_json_atomic(slot / 'state.json', state)
    if hasattr(os, 'sync'):
        os.sync()
    write_json_atomic(run_dir / 'latest.json',
                      {'slot': slot.name, 'global_step': state['global_step'], 'epoch': state['epoch']})

def load_ckpt(run_dir, model, optimizer):
    ptr = run_dir / 'latest.json'
    if not ptr.exists():
        return None
    slot = run_dir / json.loads(ptr.read_text())['slot']
    state = json.loads((slot / 'state.json').read_text())
    model.load_weights(slot / 'model.weights.h5')
    data = np.load(slot / 'optimizer.npz')
    opt_vars = optimizer.variables
    assert len(data.files) == len(opt_vars), 'optimizer variable count mismatch'
    for i, v in enumerate(opt_vars):
        arr = data[f'v{i}']
        assert tuple(v.shape) == arr.shape, f'optimizer variable {i}: shape mismatch'
        v.assign(arr)
    assert int(optimizer.iterations.numpy()) == state['global_step'], 'optimizer.iterations != global_step'
    return state

def save_best(run_dir, model, entry):
    tmp = run_dir / 'tmp_best.weights.h5'
    model.save_weights(tmp)
    os.replace(tmp, run_dir / 'best.weights.h5')
    write_json_atomic(run_dir / 'best.json', entry)

### 7.3. Training loop with resume

In [ ]:
def new_state(cfg):
    return {'config': dict(cfg), 'global_step': 0, 'epoch': 1, 'step_in_epoch': 0,
            'epoch_loss_sum': 0.0, 'epoch_loss_n': 0, 'epoch_train_sec': 0.0, 'train_sec_total': 0.0,
            'resume_count': 0, 'n_ckpts': 0, 'best_cer': float('inf'), 'best_epoch': 0,
            'bad_epochs': 0, 'history': [], 'finished': False, 'stop_reason': None}

def run_training(cfg, run_dir, train_df, val_df, ckpt_every=400, max_steps_this_call=None,
                 session_limit_min=None, log_every=200):
    """Fresh start or resume from run_dir. max_steps_this_call simulates a crash (no save)."""
    global infer  # predict() uses the global `infer`; rebind it to this model
    tf.keras.utils.set_random_seed(cfg['seed'])
    model = build_model(cfg)
    total = cfg['epochs'] * cfg['steps_per_epoch']
    sched = WarmupCosine(cfg['lr_peak'], cfg['lr_final'], cfg['warmup_steps'], total)
    opt = keras.optimizers.Adam(sched, clipnorm=cfg['clipnorm'])
    opt.build(model.trainable_variables)

    probs, _ = sampling_probs(train_df, cfg)  # oversampling weights (EXP-02)
    state = load_ckpt(run_dir, model, opt)
    if state is None:
        state = new_state(cfg)
        print('Starting fresh.')
    else:
        assert state['config'] == dict(cfg), f"config differs from checkpoint: {state['config']}"
        state['resume_count'] += 1
        print(f"Resumed: epoch {state['epoch']}, step_in_epoch {state['step_in_epoch']}, "
              f"global_step {state['global_step']}, lr {float(sched(state['global_step'])):.2e}")
    if state['finished']:
        print('Already finished:', state['stop_reason'])
        return model, opt, state

    train_step, infer = make_steps(model, opt)
    val_ds = build_dataset_bucketed(val_df, LOCAL_DIR, token_to_id, MAX_LABEL_LEN)
    t_session, steps_this_call = time.time(), 0

    while state['epoch'] <= cfg['epochs'] and not state['finished']:
        ep = state['epoch']
        epoch_df = sample_epoch(train_df, probs, cfg, ep)  # same draw after a resume (seed + epoch)
        ds = build_dataset_bucketed(epoch_df, LOCAL_DIR, token_to_id, MAX_LABEL_LEN, training=True,
                                    seed=cfg['seed'] + 1000 * ep + state['resume_count'])
        remaining = cfg['steps_per_epoch'] - state['step_in_epoch']
        t_last = time.time()
        for b in islice(ds, remaining):
            loss = float(train_step(b['image'], b['width'], b['label'], b['label_length']))
            if not math.isfinite(loss):
                raise FloatingPointError(f"non-finite loss at step {state['global_step']}; last checkpoint kept")
            now = time.time()
            state['epoch_train_sec'] += now - t_last
            state['train_sec_total'] += now - t_last
            state['global_step'] += 1
            state['step_in_epoch'] += 1
            state['epoch_loss_sum'] += loss
            state['epoch_loss_n'] += 1
            steps_this_call += 1
            if state['global_step'] % log_every == 0:
                print(f"  ep {ep} step {state['step_in_epoch']}/{cfg['steps_per_epoch']} "
                      f"(global {state['global_step']}) | loss {loss:8.2f} | "
                      f"lr {float(sched(state['global_step'])):.2e}")
            if state['global_step'] % ckpt_every == 0:
                save_ckpt(run_dir, model, opt, state)
            if max_steps_this_call and steps_this_call >= max_steps_this_call:
                print(f"SIMULATED INTERRUPT at global step {state['global_step']} (nothing saved now)")
                return model, opt, state
            if session_limit_min and (time.time() - t_session) / 60 >= session_limit_min:
                save_ckpt(run_dir, model, opt, state)
                print('Session time limit reached: checkpoint saved, stopping cleanly.')
                return model, opt, state
            t_last = time.time()  # excludes checkpoint time from the training clock

        if state['step_in_epoch'] != cfg['steps_per_epoch']:
            raise RuntimeError(f"epoch {ep}: the pipeline gave {state['step_in_epoch']} of "
                               f"{cfg['steps_per_epoch']} steps; increase draw_factor")
        # ---- end of epoch: validate on the fixed subset ----
        refs, hyps = predict(val_ds)
        m, _ = evaluate_sequences(refs, hyps, families)
        entry = {'epoch': ep, 'global_step': state['global_step'],
                 'train_loss': state['epoch_loss_sum'] / max(1, state['epoch_loss_n']),
                 'val_cer': m['cer'], 'val_seq_acc': m['seq_acc'], 'val_cer_shape': m['cer_shape'],
                 'lr_end': float(sched(state['global_step'])),
                 'epoch_train_min': state['epoch_train_sec'] / 60}
        entry = {k: float(v) for k, v in entry.items()}
        state['history'].append(entry)
        if m['cer'] < state['best_cer']:
            state['best_cer'], state['best_epoch'], state['bad_epochs'] = float(m['cer']), ep, 0
            save_best(run_dir, model, entry)
            tag = ' *best*'
        else:
            state['bad_epochs'] += 1
            tag = f" (no improvement x{state['bad_epochs']})"
        print(f"EPOCH {ep} | loss {entry['train_loss']:.2f} | val CER {m['cer']:.4f} | "
              f"seq_acc {m['seq_acc']:.3f} | cer_shape {m['cer_shape']:.4f} | "
              f"{entry['epoch_train_min']:.1f} min{tag}")
        write_json_atomic(run_dir / 'history.json', state['history'])

        state['epoch'] += 1
        state['step_in_epoch'] = 0
        state['epoch_loss_sum'], state['epoch_loss_n'], state['epoch_train_sec'] = 0.0, 0, 0.0
        if ep >= cfg['early_stop_from_epoch'] and state['bad_epochs'] >= cfg['patience']:
            state['finished'], state['stop_reason'] = True, 'early_stop'
        elif state['epoch'] > cfg['epochs']:
            state['finished'], state['stop_reason'] = True, 'epochs_done'
        save_ckpt(run_dir, model, opt, state)

    print('Finished:', state['stop_reason'])
    return model, opt, state

## 8. Launch EXP-02

Logs print every 200 steps. Things to check in the first minutes:
- **Speed:** EXP-00 needed about 3.2 min per 200 steps (about 29.5 min per epoch). Expected to be similar: the model and the batch policy are the same; the image-width mix shifts slightly.
- **Epoch 1:** the validation CER should be on the order of 0.03 (EXP-00: 0.0317). Compare epoch by epoch with EXP-00 in section 9.1.

In [ ]:
model, opt, state = run_training(CFG, RUN_DIR, parts['train'], val_sub,
                                 ckpt_every=400, session_limit_min=None)

Starting fresh.
  ep 1 step 200/1830 (global 200) | loss   153.40 | lr 1.00e-03
  ep 1 step 400/1830 (global 400) | loss   114.75 | lr 1.00e-03
  ep 1 step 600/1830 (global 600) | loss    69.01 | lr 9.98e-04
  ep 1 step 800/1830 (global 800) | loss    31.59 | lr 9.96e-04
  ep 1 step 1000/1830 (global 1000) | loss    15.90 | lr 9.93e-04
  ep 1 step 1200/1830 (global 1200) | loss     8.09 | lr 9.88e-04
  ep 1 step 1400/1830 (global 1400) | loss     5.95 | lr 9.83e-04
  ep 1 step 1600/1830 (global 1600) | loss     4.28 | lr 9.77e-04
  ep 1 step 1800/1830 (global 1800) | loss     3.09 | lr 9.70e-04
EPOCH 1 | loss 63.24 | val CER 0.0291 | seq_acc 0.555 | cer_shape 0.0238 | 34.5 min *best*
  ep 2 step 170/1830 (global 2000) | loss     6.34 | lr 9.63e-04
  ep 2 step 370/1830 (global 2200) | loss     3.56 | lr 9.54e-04
  ep 2 step 570/1830 (global 2400) | loss     5.82 | lr 9.44e-04
  ep 2 step 770/1830 (global 2600) | loss     3.86 | lr 9.34e-04
  ep 2 step 970/1830 (global 2800) | loss     2

## 9. Results

### 9.1. Training curves vs EXP-00 (fixed 2,000-sample validation subset)

In [ ]:
h00 = pd.DataFrame(json.loads((CKPT_DIR / 'exp00' / 'history.json').read_text()))
h02 = pd.DataFrame(json.loads((RUN_DIR / 'history.json').read_text()))
cols = ['epoch', 'val_cer', 'val_seq_acc', 'val_cer_shape', 'epoch_train_min']
cmp = h00[cols].merge(h02[cols], on='epoch', how='right', suffixes=('_exp00', '_exp02'))
display(cmp.round(4))

,epoch,val_cer_exp00,val_seq_acc_exp00,val_cer_shape_exp00,epoch_train_min_exp00,val_cer_exp02,val_seq_acc_exp02,val_cer_shape_exp02,epoch_train_min_exp02
0,1.0,0.0317,0.5920,0.0257,29.4756,0.0291,0.5550,0.0238,34.4857
1,2.0,0.0156,0.7605,0.0125,29.4351,0.0139,0.7765,0.0115,31.7707
2,3.0,0.0107,0.8150,0.0089,29.4188,0.0107,0.8210,0.0087,31.8881
3,4.0,0.0089,0.8450,0.0074,29.7121,0.0082,0.8490,0.0068,31.5433
4,5.0,0.0082,0.8530,0.0070,29.6914,0.0071,0.8700,0.0058,31.3156
5,6.0,0.0077,0.8605,0.0062,29.6023,0.0069,0.8745,0.0056,31.3085
6,7.0,0.0075,0.8655,0.0061,29.5250,0.0065,0.8800,0.0053,31.3750
7,8.0,0.0073,0.8695,0.0060,29.5260,0.0064,0.8825,0.0052,31.2998


### 9.2. Row for experiments/results.csv

In [ ]:
h = pd.DataFrame(state['history'])
b = h.loc[h['val_cer'].idxmin()]
row = {'exp_id': 'EXP-02', 'date': time.strftime('%Y-%m-%d'), 'epochs_run': len(h),
       'best_epoch': int(b['epoch']), 'lr_schedule': 'warmup 200 + cosine 1e-3 to 1e-5',
       'batch_policy': 'bucketed; 48000 px; max 64', 'val_subset_cer': round(b['val_cer'], 4),
       'seq_acc': round(b['val_seq_acc'], 4), 'cer_shape': round(b['val_cer_shape'], 4),
       'train_time_h': round(state['train_sec_total'] / 3600, 2),
       'notes': (f"Oversampling with replacement K={CFG['oversample_k']} of images with natural/slur/gracenote tokens; "
                 f"draw x{CFG['draw_factor']}; Adam clipnorm 5; val subset 2000; early stop patience 3 from epoch 4")}
print(pd.DataFrame([row]).to_csv(index=False))

exp_id,date,epochs_run,best_epoch,lr_schedule,batch_policy,val_subset_cer,seq_acc,cer_shape,train_time_h,notes
EXP-02,2026-11-18,8,8,warmup 200 + cosine 1e-3 to 1e-5,bucketed; 48000 px; max 64,0.0064,0.8825,0.0052,4.25,Oversampling with replacement K=3 of images with natural/slur/gracenote tokens; draw x1.1; Adam clipnorm 5; val subset 2000; early stop patience 3 from epoch 4

